# Analytics: Aggregation, Joins, Windows, Cohorts and Dashboards

Notebook for **Chapter 4** of the *ways* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ways/ch04.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
# Setup: the files used in this chapter are downloaded next to the notebook.
import os, urllib.request
os.makedirs("data", exist_ok=True)
for f in ["customers.csv", "orders.csv", "products.csv", "shopkart.db"]:
    if not os.path.exists("data/" + f):
        urllib.request.urlretrieve("https://dadipavan.github.io/Learning/study/ways/data/" + f, "data/" + f)


## 4.1 Aggregation: group, then summarise

Aggregation is the question "for each X, what is the total or average of Y?". Three parts: the *grouping key* (category), the *measure* (revenue) and the *function* (sum). Everything else is syntax.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
from collections import defaultdict

o, c, p = prep.load_clean(DATA)
con = prep.clean_db(DATA)

sql = con.execute("""SELECT category, ROUND(SUM(revenue)) AS revenue, COUNT(*) AS orders
                     FROM orders GROUP BY category ORDER BY revenue DESC""").fetchall()
pdf = (o.groupby("category").agg(revenue=("revenue", "sum"), orders=("order_id", "count"))
         .sort_values("revenue", ascending=False).round(0))

tot, cnt = defaultdict(float), defaultdict(int)
for cat, rev in zip(o["category"], o["revenue"]):
    tot[cat] += rev
    cnt[cat] += 1
py = sorted(((k, round(v), cnt[k]) for k, v in tot.items()), key=lambda t: -t[1])

print(pdf.to_string())
print("all three equal:", [(a, int(b), n) for a, b, n in sql] == [(k, int(r.revenue), int(r.orders)) for k, r in pdf.iterrows()] == [(a, int(b), n) for a, b, n in py])

revenue  orders
category                      
Fashion      5247054.0    4043
Electronics  4863137.0    2735
Grocery      4055843.0    7648
Home         2362981.0    2830
Beauty        820901.0    2739
all three equal: True


## 4.2 Time: months, growth and seasonality

Time questions need a period key (month), a measure, and often a comparison with the previous period. We compute monthly revenue and month-on-month growth.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
o, c, p = prep.load_clean(DATA)
con = prep.clean_db(DATA)

q = """SELECT substr(order_date, 1, 7) AS month, ROUND(SUM(revenue)) AS revenue
       FROM orders GROUP BY 1 ORDER BY 1"""
sql = con.execute(q).fetchall()
m = o.groupby(o.order_date.dt.strftime("%Y-%m")).revenue.sum().round()
print("sql == pandas:", all(abs(a[1] - b) <= 1 for a, b in zip(sql, m)) and len(sql) == len(m))
g = (m.pct_change() * 100).round(1)
print(pd_out := m.astype(int).to_frame("revenue").assign(growth_pct=g).to_string())
print("best month:", m.idxmax())

sql == pandas: True
            revenue  growth_pct
order_date                     
2024-01      863567         NaN
2024-02     1075009        24.5
2024-03     1243734        15.7
2024-04     1441432        15.9
2024-05     1622693        12.6
2024-06     1716713         5.8
2024-07     1741960         1.5
2024-08     1796818         3.1
2024-09     1479687       -17.6
2024-10     1422043        -3.9
2024-11     1635052        15.0
2024-12     1311209       -19.8
best month: 2024-08


## 4.3 Joins: combining tables, and the fan-out trap

A join matches rows from two tables on a key. **Inner join** keeps matches only; **left join** keeps every left row, with missing on the right when no match. Which one you choose changes the row count, so always check the count before and after.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
o, c, p = prep.load_clean(DATA)
con = prep.clean_db(DATA)

print(con.execute("""SELECT c.segment, COUNT(*) AS orders, ROUND(SUM(o.revenue)) AS revenue
                     FROM orders o JOIN customers c ON c.customer_id = o.customer_id
                     GROUP BY c.segment ORDER BY revenue DESC""").fetchall())
j = o.merge(c, on="customer_id", how="inner")
print("rows before/after join:", len(o), len(j))
print(j.groupby("segment").agg(orders=("order_id", "count"), revenue=("revenue", "sum")).round(0).sort_values("revenue", ascending=False).to_dict("index"))

never_sql = con.execute("""SELECT COUNT(*) FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
                           WHERE o.customer_id IS NULL""").fetchone()[0]
never_pd = (~c.customer_id.isin(o.customer_id)).sum()
print("customers who never ordered:", never_sql, never_pd)

[('Regular', 11317, 9875013.0), ('Premium', 6235, 5422564.0), ('Student', 2443, 2052339.0)]
rows before/after join: 19995 19995
{'Regular': {'orders': 11317, 'revenue': 9875012.0}, 'Premium': {'orders': 6235, 'revenue': 5422564.0}, 'Student': {'orders': 2443, 'revenue': 2052339.0}}
customers who never ordered: 785 785


In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
import pandas as pd
o, c, p = prep.load_clean(DATA)
promo = pd.DataFrame({"category": ["Grocery", "Grocery", "Fashion", "Fashion", "Home", "Home", "Beauty", "Beauty", "Electronics", "Electronics"],
                      "campaign": ["A", "B"] * 5})
bad = o.merge(promo, on="category")
print("rows:", len(o), "->", len(bad), "| revenue:", round(o.revenue.sum()), "->", round(bad.revenue.sum()))
good = o.merge(promo.groupby("category").size().rename("n_campaigns").reset_index(), on="category")
print("one row per key keeps rows:", len(good), "| revenue:", round(good.revenue.sum()))
try:
    o.merge(promo, on="category", validate="m:1")
except Exception as e:
    print("validate caught it:", type(e).__name__)

rows: 19995 -> 39990 | revenue: 17349916 -> 34699831
one row per key keeps rows: 19995 | revenue: 17349916
validate caught it: MergeError


## 4.4 Window functions: compare a row to its neighbours

An aggregate collapses rows; a window function keeps every row and adds a calculation over a "window" of related rows. The typical cases: running totals, ranks within groups, previous-period comparisons and moving averages.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
o, c, p = prep.load_clean(DATA)
con = prep.clean_db(DATA)

print("--- SQL: running total, previous month, top product per category")
q = """
WITH m AS (SELECT substr(order_date,1,7) AS month, SUM(revenue) AS rev FROM orders GROUP BY 1),
     w AS (SELECT month, rev, SUM(rev) OVER (ORDER BY month) AS running, LAG(rev) OVER (ORDER BY month) AS prev FROM m)
SELECT month, ROUND(rev), ROUND(running), ROUND(prev) FROM w
WHERE month IN ('2024-01','2024-02','2024-12')"""
for r in con.execute(q): print(r)

q2 = """
SELECT category, product, ROUND(rev) FROM (
  SELECT category, product, SUM(revenue) AS rev,
         ROW_NUMBER() OVER (PARTITION BY category ORDER BY SUM(revenue) DESC) AS rn
  FROM orders GROUP BY category, product) WHERE rn = 1 ORDER BY category"""
sql_top = con.execute(q2).fetchall()
for r in sql_top: print(r)

print("--- pandas")
m = o.groupby(o.order_date.dt.strftime("%Y-%m")).revenue.sum()
w = m.to_frame("rev").assign(running=m.cumsum(), prev=m.shift(1)).round(0)
print(w.loc[["2024-01", "2024-02", "2024-12"]].to_string())
g = o.groupby(["category", "product"]).revenue.sum().round(0).reset_index()
top = g.loc[g.groupby("category").revenue.idxmax()].sort_values("category")
print("top product per category equal:", all((a, b) == (a2, b2) and abs(r - r2) <= 1 for (a, b, r), (a2, b2, r2) in zip(sql_top, top.itertuples(index=False))))

--- SQL: running total, previous month, top product per category
('2024-01', 863567.0, 863567.0, None)
('2024-02', 1075009.0, 1938576.0, 863567.0)
('2024-12', 1311209.0, 17349916.0, 1635053.0)
('Beauty', 'Perfume', 263097.0)
('Electronics', 'Mouse', 1822785.0)
('Fashion', 'Socks 3-pack', 2241445.0)
('Grocery', 'Atta 5kg', 1739550.0)
('Home', 'Bedsheet', 1123093.0)
--- pandas
                  rev     running       prev
order_date                                  
2024-01      863567.0    863567.0        NaN
2024-02     1075009.0   1938576.0   863567.0
2024-12     1311209.0  17349916.0  1635052.0
top product per category equal: True


In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
import numpy as np
o, c, p = prep.load_clean(DATA)
con = prep.clean_db(DATA)

sql = con.execute("""
SELECT d, ROUND(AVG(n) OVER (ORDER BY d ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 4)
FROM (SELECT order_date AS d, COUNT(*) AS n FROM orders GROUP BY order_date) ORDER BY d""").fetchall()
daily = o.groupby("order_date").size()
ma = daily.rolling(7, min_periods=1).mean().round(4)
print("days:", len(daily), "| equal:", np.allclose([r[1] for r in sql], ma.values))
print("last day raw", daily.iloc[-1], "| 7-day average", round(ma.iloc[-1], 2))

loop = []
vals = list(daily.values)
for i in range(len(vals)):
    window = vals[max(0, i - 6): i + 1]
    loop.append(sum(window) / len(window))
print("plain Python loop equal:", np.allclose(loop, ma.values))

days: 366 | equal: True
last day raw 68 | 7-day average 49.86
plain Python loop equal: True


## 4.5 Cohorts and retention

A **cohort** is a group of customers who share a starting point, here the month of their first order in the data. **Retention** asks: of the customers in the cohort, what share ordered again k months later? It separates "we have more orders because we have more customers" from "customers stay".

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
import numpy as np, pandas as pd
o, c, p = prep.load_clean(DATA)
con = prep.clean_db(DATA)

q = """
WITH f AS (SELECT customer_id, MIN(substr(order_date,1,7)) AS cohort FROM orders GROUP BY 1),
     a AS (SELECT DISTINCT customer_id, substr(order_date,1,7) AS m FROM orders)
SELECT cohort,
       (CAST(substr(m,1,4) AS INT)*12 + CAST(substr(m,6,2) AS INT)) -
       (CAST(substr(cohort,1,4) AS INT)*12 + CAST(substr(cohort,6,2) AS INT)) AS k,
       COUNT(*) AS n
FROM a JOIN f USING (customer_id) GROUP BY 1, 2"""
sql = pd.DataFrame(con.execute(q).fetchall(), columns=["cohort", "k", "n"]).pivot(index="cohort", columns="k", values="n")

o["m"] = o.order_date.dt.to_period("M")
first = o.groupby("customer_id").m.min().rename("cohort")
x = o.join(first, on="customer_id")
x["k"] = (x.m - x.cohort).apply(lambda v: v.n)
mat = x.groupby(["cohort", "k"]).customer_id.nunique().unstack()
print("SQL matrix equals pandas matrix:", np.array_equal(np.nan_to_num(sql.values), np.nan_to_num(mat.values)))

ret = (mat.div(mat[0], axis=0) * 100).round(0)
print(ret.loc[:, 0:4].head(6).to_string())
print("average month-1 retention, cohorts Jan-Oct:", round(ret.loc[:"2024-10", 1].mean(), 1), "%")

SQL matrix equals pandas matrix: True
k            0     1     2     3     4
cohort                                
2024-01  100.0  53.0  55.0  59.0  60.0
2024-02  100.0  52.0  48.0  53.0  50.0
2024-03  100.0  49.0  53.0  47.0  48.0
2024-04  100.0  45.0  46.0  45.0  33.0
2024-05  100.0  46.0  47.0  43.0  39.0
2024-06  100.0  40.0  39.0  31.0  30.0
average month-1 retention, cohorts Jan-Oct: 43.2 %


## 4.6 Seeing it: six charts and the design checklist

A chart is an argument: it should make one comparison easy. The choice of form follows the question, not the other way round.

## 4.7 Speed and readability: when the tool matters

All four tools got the same numbers. They differ in speed on larger data, in how the logic reads, and in who can maintain it. To feel the speed, we repeat the category revenue 20 times (400,000 rows) and time three methods. Exact seconds depend on your machine, so we print only which is faster, which is stable.

In [ ]:
DATA = "data/"
import sys, time, sqlite3
sys.path.insert(0, DATA)
import prep
import pandas as pd
o, c, p = prep.load_clean(DATA)
big = pd.concat([o[["category", "revenue"]]] * 20, ignore_index=True)

t = time.perf_counter()
tot = {}
for cat, rev in zip(big["category"], big["revenue"]):
    tot[cat] = tot.get(cat, 0.0) + rev
t_loop = time.perf_counter() - t

t = time.perf_counter()
pdres = big.groupby("category").revenue.sum()
t_pd = time.perf_counter() - t

con = sqlite3.connect(":memory:")
big.to_sql("big", con, index=False)
sres = dict(con.execute("SELECT category, SUM(revenue) FROM big GROUP BY category").fetchall())

print("rows:", len(big))
print("same answer:", all(abs(tot[k] - pdres[k]) < 1 and abs(sres[k] - pdres[k]) < 1 for k in tot))
print("pandas faster than loop:", t_pd < t_loop)

rows: 399900
same answer: True
pandas faster than loop: True


In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
o, c, p = prep.load_clean(DATA)
con = prep.clean_db(DATA)
sql = con.execute("SELECT category, ROUND(SUM(revenue) * 100.0 / SUM(SUM(revenue)) OVER (), 1) FROM orders GROUP BY category ORDER BY 2 DESC").fetchall()
g = o.groupby("category").revenue.sum()
pdv = (g / g.sum() * 100).round(1).sort_values(ascending=False)
print(sql)
print(pdv.to_dict())

[('Fashion', 30.2), ('Electronics', 28.0), ('Grocery', 23.4), ('Home', 13.6), ('Beauty', 4.7)]
{'Fashion': 30.2, 'Electronics': 28.0, 'Grocery': 23.4, 'Home': 13.6, 'Beauty': 4.7}


## Chapter recap

## Practice set

Problem 4.8.1 Which category has the highest net revenue, and what share of total revenue is it?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ways/ch04.html